# 03 · Reward shaping and learning stability: TD3 vs DDPG

How does the reward the agent *trains on* change learning stability, and does it change how the finished policy copes with failures? We train **TD3 and DDPG** with identical hyperparameters on five reward variants (`default`, `dist_only`, `sparse`, `progress`, `scaled_x10`) across 5 seeds, then stress-test every solved policy.

Everything is judged on the **unshaped task**: summed negative distance to the target (higher is better) and success rate (fingertip within 5 cm at the end). Shaped training returns are never compared, because each variant uses a different reward scale.

The full sweep takes hours, so this notebook does not run it: it loads the committed results in `plots/`. To reproduce, run `python scripts/sweep.py`, `python scripts/sweep_analysis.py`, `python scripts/shaping_robustness.py`. Set `DFC_QUICK=1` for a tiny smoke-test sweep.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "envs" / "stress_wrappers.py").exists())
sys.path.insert(0, str(ROOT))

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

QUICK = os.environ.get("DFC_QUICK", "0") == "1"  # DFC_QUICK=1 -> tiny sweep, for smoke-testing only
PLOTS = ROOT / "plots" / ("quick" if QUICK else "")
PLOTS.mkdir(parents=True, exist_ok=True)
print("project root :", ROOT.name)
print("quick mode   :", QUICK)

## Load results

In [ ]:
from scripts.train import load_config
from scripts.sweep import load_sweep_config, quick_config, run_sweep
from scripts.sweep_analysis import plot_learning_curves, plot_metric_comparison, write_sweep_outputs
from scripts.shaping_robustness import plot_robustness, run_robustness

train_cfg = load_config()
sweep_cfg = load_sweep_config()
threshold = sweep_cfg["solve_threshold"]

if QUICK:
    sweep_cfg = quick_config(sweep_cfg)           # 2 algos x 2 variants x 1 seed, ~800 steps
    run_sweep(sweep_cfg, train_cfg, log=print)    # resumable
    write_sweep_outputs(sweep_cfg["runs_dir"], PLOTS, threshold)
    try:
        run_robustness(sweep_cfg, train_cfg, PLOTS, workers=1)
    except ValueError as err:                     # quick policies rarely reach the solve threshold
        print("robustness skipped in quick mode:", err)
else:
    needed = ["sweep_eval_curves.csv", "sweep_summary.csv", "sweep_random_baseline.json"]
    missing = [n for n in needed if not (PLOTS / n).exists()]
    assert not missing, f"missing {missing} in plots/: run `python scripts/sweep.py` then `python scripts/sweep_analysis.py`"

curves = pd.read_csv(PLOTS / "sweep_eval_curves.csv")
summary = pd.read_csv(PLOTS / "sweep_summary.csv")
random_baseline = json.loads((PLOTS / "sweep_random_baseline.json").read_text())["mean_distance_return"]
print(f"{curves.groupby(['algo', 'variant', 'seed']).ngroups} runs loaded; random policy = {random_baseline:.2f}")

## Stability table
Means over seeds with 95% bootstrap CIs (crude with 5 seeds). `n_reached` = seeds that reached the solve threshold.

In [ ]:
cols = ["algo", "variant", "n_seeds", "n_reached", "final_return_mean", "final_return_lo", "final_return_hi",
        "final_success_mean", "auc_mean", "steps_to_threshold_mean", "max_drawdown_mean"]
display(summary[cols].round(3))

## Learning curves
Thin lines are individual seeds, thick lines the mean, bands the 95% bootstrap CI.

In [ ]:
fig = plot_learning_curves(curves, random_baseline, threshold)
plt.show()

## Metric comparison

In [ ]:
fig = plot_metric_comparison(summary)
plt.show()
diffs_path = PLOTS / "sweep_algo_differences.csv"
if diffs_path.exists():
    display(pd.read_csv(diffs_path).round(3))

## Robustness of the solved policies
Each solved final policy is stress-tested with a reduced IPQC battery. Policies that never reached the solve threshold are excluded (see the eligibility table), never scored.

In [ ]:
rob_paths = {n: PLOTS / n for n in ["shaping_eligibility.csv", "shaping_robustness_summary.csv", "shaping_boundaries_summary.csv"]}
if all(p.exists() for p in rob_paths.values()):
    display(pd.read_csv(rob_paths["shaping_eligibility.csv"]))
    rob_summary = pd.read_csv(rob_paths["shaping_robustness_summary.csv"])
    if rob_summary.empty:
        print("no policy reached the solve threshold, so there is nothing to compare")
    else:
        plot_robustness(rob_summary)
        plt.show()
        display(pd.read_csv(rob_paths["shaping_boundaries_summary.csv"]).round(3))
else:
    print("no robustness results yet: run `python scripts/shaping_robustness.py`")

## Findings

*(written after the full run; see the README for the discussion)*